### 1. Discrete‑Time Dynamical System (Explicit Euler)

Let the state at step $k$ be $\mathbf{x}_k \in \mathbb{R}^n$, and let the step size be $h > 0$.  
Given a fixed permutation matrix $P$ (e.g., swapping coordinates $i$ and $j$) and a deterministic binary schedule $\{s_k\}_{k=0}^{\infty} \subset \{0,1\}$, the Euler iteration is:

$$
\mathbf{x}_{k+1} = \mathbf{x}_k + h \cdot \mathbf{F}^{(s_k)}(\mathbf{x}_k),
$$

where

$$
\mathbf{F}^{(s_k)}(\mathbf{x}_k) = \big[(1 - s_k) I + s_k P\big] \mathbf{F}(\mathbf{x}_k).
$$

Expanding the bracket:

$$
\boxed{\mathbf{x}_{k+1} = \mathbf{x}_k + h \Big( (1 - s_k)\,\mathbf{F}(\mathbf{x}_k) + s_k \, P\mathbf{F}(\mathbf{x}_k) \Big)}
$$

---

#### Explicit 2D Example
Let $n=2$, $\mathbf{x} = (x, y)^\top$, $\mathbf{F} = (F_x, F_y)^\top$, and $P$ be the full swap $\begin{pmatrix}0&1\\1&0\end{pmatrix}$.  
Then:

- If $s_k = 0$ (no swap):
  $$
  \begin{cases}
  x_{k+1} = x_k + h\, F_x(x_k, y_k) \\
  y_{k+1} = y_k + h\, F_y(x_k, y_k)
  \end{cases}
  $$

- If $s_k = 1$ (swap):
  $$
  \begin{cases}
  x_{k+1} = x_k + h\, F_y(x_k, y_k) \\
  y_{k+1} = y_k + h\, F_x(x_k, y_k)
  \end{cases}
  $$

---

### 2. General $n$-D Algorithm (Pseudo‑code)

For arbitrary dimension $n$ and a permutation $P$ (which may be a transposition or a longer cycle), the iteration is:

```text
Input: x_0 (initial state), F (function handle), P (n x n permutation matrix),
       h (step size), S = [s_0, s_1, ..., s_{N-1}] (user‑supplied binary array)
Output: trajectory x_0, x_1, ..., x_N

for k = 0 to N-1 do:
    F_val = F(x_k)
    if S[k] == 0:
        x_{k+1} = x_k + h * F_val
    else:
        x_{k+1} = x_k + h * (P @ F_val)   # matrix‑vector multiplication
end for
```

**Determinism guarantee:** Since $S$ is fixed *a priori* and no random number generator is used, every run with the same $S$, $h$, and $\mathbf{x}_0$ yields identical results.

---

### 3. Discrete Coupling Coefficient

Over $N$ iterations, the coupling coefficient exactly matches your empirical frequency definition:

$$
\boxed{C_N = \frac{1}{N} \sum_{k=0}^{N-1} s_k = \frac{\text{number of } 1\text{s in } S}{N}}
$$

This scalar lies in $[0,1]$. A schedule of all zeros gives $C_N = 0$ (uncoupled), all ones gives $C_N = 1$ (fully swapped at every step), and a mixture gives intermediate coupling strength.

---

### 4. Extending to Higher‑Order Deterministic Integrators (e.g., RK4)

Because the swap operator $L_{s} = (1-s)I + sP$ is a **linear operator** acting on the vector field, it commutes with scalar multiplication and addition. Therefore, you can apply it directly inside any Runge–Kutta stage without breaking determinism.

For RK4, at step $k$ with binary value $s = s_k$:

1. Compute the base field $\mathbf{F}_0 = \mathbf{F}(\mathbf{x}_k)$.
2. Compute the swapped field $\mathbf{F}_1 = L_s \mathbf{F}_0$.
3. Compute intermediate stages using the *same* linear operator:
   $$
   \begin{aligned}
   \mathbf{k}_1 &= L_s\, \mathbf{F}(\mathbf{x}_k), \\
   \mathbf{k}_2 &= L_s\, \mathbf{F}\!\left(\mathbf{x}_k + \frac{h}{2}\mathbf{k}_1\right), \\
   \mathbf{k}_3 &= L_s\, \mathbf{F}\!\left(\mathbf{x}_k + \frac{h}{2}\mathbf{k}_2\right), \\
   \mathbf{k}_4 &= L_s\, \mathbf{F}\!\left(\mathbf{x}_k + h\,\mathbf{k}_3\right),
   \end{aligned}
   $$
   $$
   \mathbf{x}_{k+1} = \mathbf{x}_k + \frac{h}{6}(\mathbf{k}_1 + 2\mathbf{k}_2 + 2\mathbf{k}_3 + \mathbf{k}_4).
   $$

This preserves the deterministic nature and maintains the same coupling operator throughout the substeps of a single macro‑iteration.

---

### 5. Constructing Deterministic Schedules

To remain strictly non‑stochastic, the user‑supplied array $S$ can be generated by any deterministic rule, such as an **arbitrary hard‑coded list**: $S = [0,1,1,0,1,0,\dots]$.

The choice of $S$ directly controls $C_N$, allowing you to dial the coupling strength deterministically.

---

### 6. Stability and Step‑Size Consideration

Since applying $P$ changes the direction of the vector field components, the Lipschitz constant of $\mathbf{F}^{(s)}$ may differ from that of $\mathbf{F}$. For explicit Euler, ensure the step size $h$ satisfies stability bounds for **both** the swapped and unswapped regimes, i.e.,

$$
h < \min_{s \in \{0,1\}} \frac{2}{\lambda_{\max}\left( J^{(s)} \right)},
$$

where $J^{(s)}$ is the Jacobian of $\mathbf{F}^{(s)}$, to maintain deterministic numerical stability without relying on adaptive random step sizes.